# 5. Nested cross-validation and the classic genomics leak

**Learning goals**: why tuning and evaluating on the same folds is optimistic, and why
selecting genes on the full data before cross-validating can manufacture accuracy from pure noise.

Ambroise and McLachlan (2002) showed that many published gene-expression classifiers reported
near-perfect accuracy because the genes were chosen using all samples, including those later used
as "test" data. The cure is to put **every** data-dependent step (gene filtering, scaling, batch
correction, tuning) inside the training folds.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(7)

## Pure noise: 200 samples, 5,000 random genes, a random response

In [2]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_predict
n, p = 200, 5000
X = rng.normal(size=(n, p)); y = rng.normal(size=n)          # no signal at all
cv = KFold(5, shuffle=True, random_state=0)

# WRONG: pick the 50 genes most correlated with y using all samples, then cross-validate
corr = np.abs((X - X.mean(0)).T @ (y - y.mean())) / (X.std(0) * y.std() * n)
top = np.argsort(corr)[-50:]
pred_leak = cross_val_predict(RidgeCV(alphas=np.logspace(-2, 4, 20)), X[:, top], y, cv=cv)

# RIGHT: choose the 50 genes inside each training fold
pred_ok = np.zeros(n)
for tr, te in cv.split(X):
    c = np.abs((X[tr] - X[tr].mean(0)).T @ (y[tr] - y[tr].mean()))
    t = np.argsort(c)[-50:]
    pred_ok[te] = RidgeCV(alphas=np.logspace(-2, 4, 20)).fit(X[tr][:, t], y[tr]).predict(X[te][:, t])
print(f"leaky selection : cross-validated r = {np.corrcoef(pred_leak, y)[0, 1]:.2f}")
print(f"honest selection: cross-validated r = {np.corrcoef(pred_ok, y)[0, 1]:.2f}   (truth: 0)")

leaky selection : cross-validated r = 0.77
honest selection: cross-validated r = 0.13   (truth: 0)


The leaky pipeline "predicts" pure noise with a sizeable correlation. In this project the gene filter
uses expression **variance** (it never sees the response), but it is still computed inside each
outer training fold, along with the batch correction and standardisation (`drugresponse.models.Design`).

## Nested CV: tuning is part of the model
If $\lambda$ is chosen to minimise the cross-validation error and that same error is reported,
the report is biased downwards: the minimum of many noisy estimates is too low. Nested CV tunes
$\lambda$ in an inner loop on the training fold only and reports the outer-fold error.

In [3]:
from sklearn.linear_model import LassoCV
X = rng.normal(size=(150, 2000)); y = X[:, :5] @ np.full(5, 0.4) + rng.normal(size=150)
outer = KFold(5, shuffle=True, random_state=1)
m = LassoCV(cv=outer, alphas=40).fit(X, y)
non_nested = m.mse_path_.mean(1).min()
nested = []
for tr, te in outer.split(X):
    mm = LassoCV(cv=5, alphas=40).fit(X[tr], y[tr])
    nested.append(np.mean((y[te] - mm.predict(X[te])) ** 2))
print(f"non-nested CV MSE (tuned and scored on the same folds): {non_nested:.3f}")
print(f"nested CV MSE (tuned inside, scored outside):           {np.mean(nested):.3f}")

non-nested CV MSE (tuned and scored on the same folds): 1.606
nested CV MSE (tuned inside, scored outside):           1.678


The gap is usually modest for one tuning parameter, and it grows with the number of choices tried
(genes kept, penalty, mixing weight, model family). The project's headline numbers are all outer-fold.

**Key takeaways**
* Every step that looks at the data belongs inside the training folds.
* Selecting features on all samples can create accuracy out of nothing.
* Report nested-CV error; the inner CV is for tuning only.